# DiagnosticIQ source step 3: data formatting

Adds the `asset` field (parsed from the rule name) and splits each rule's observations into a list, rewriting the consistency-check outputs in place.

Run this once on fresh consistency-check output: both steps modify the files in place, so running it twice corrupts them.

In [ ]:
import os
import re
import json
from datetime import datetime
from typing import Optional, List, Callable, Any
from abc import abstractmethod, ABC
from functools import partial
import copy

In [ ]:
#parameter 

from dataset_utils.paths import get_dataset_dir

extract_fol = get_dataset_dir() / "extracted"
# The four files written by the consistency check notebook.
dataset_locs = [
    os.path.join(extract_fol, x)
    for x in ["observations.rules.json", "no_observations.no_rules.json", "only_observations.json", "only_rules.json"]
]

In [ ]:
def save_json(data:dict, loc:str) -> None:
    with open(loc, "w") as f0:
        json.dump(data, f0)

def load_json(loc:str) -> dict:
    with open(loc, "r") as f0:
        return json.load(f0)
    
def save_json_with_outputfile(data:dict, file_path:str, output_file:Optional[str] = None) -> None:
    if not output_file:
        output_file = file_path
    save_json(data, output_file)
    

def add_asset(file_path:str, output_file:Optional[str]=None) -> None:
    data = load_json(file_path)

    for d in data["rule_set"]:
        if  " - " in d['name'].strip() and " – " in d["name"].strip():
            use = {" - ": d['name'].strip().find(" - "), " – ":d["name"].strip().find(" – ")}
            use_val = min(use, key=use.get)
            d["asset"] = d['name'].strip().split(use_val)[0]
        elif " - " in d['name'].strip():
            d["asset"] = d['name'].strip().split(" - ")[0]
        elif " – " in d["name"].strip():
            d["asset"] = d['name'].strip().split(" – ")[0]
        else:
            d["asset"] = None
            print(d['name'].strip(), d["#n"])

    save_json_with_outputfile(data,file_path, output_file)

def sep_observations(file_path:str, output_file:Optional[str]=None) -> None:
    data = load_json(file_path)

    for d in data["rule_set"]:
        if d["display_text"]["observations"]:
            d["display_text"]["observations"] = d["display_text"]["observations"].split("\n -")
            d["display_text"]["observations"][0] = d["display_text"]["observations"][0][2:]
            d["display_text"]["observations"] = [x.strip().replace("\n", "") for x in d["display_text"]["observations"]]

    save_json_with_outputfile(data,file_path, output_file)

def handle_multiple_datasets(locs:List[str], func:Callable[[str, Optional[str]], None], sep_out:Optional[Callable[[str],str]]=False) -> None:
    for f in locs:
        if sep_out:
            #f"{f.split('.')[0]}.{func.__name__}.{f.split('.')[-1]}"
            func(f, sep_out(f))
        else:
            func(f, None)


In [ ]:
handle_multiple_datasets(dataset_locs, add_asset)

In [ ]:
handle_multiple_datasets(dataset_locs, sep_observations)